# baseline v3

이 베이스라인 코드는 `사전학습 모델 로드`, `배치 학습`, `파인튜닝`, `양자화`, `PEFT` 등이 적용된 버전입니다.

윈도우 데스크탑의 RTX 5060 ti GPU 환경에서 개발되었습니다.

# 환경 준비

개발 환경에 필요한 라이브러리 버전을 고정하고 최신 버전으로 라이브러리를 업데이트합니다.

- 아래 셀 실행
- ipykernel 설치
- 아래 셀 다시 실행 : 무한 로딩 시 restart
- hello 출력시 torch 설치

In [1]:
!pip install pandas

In [ ]:
!pip install ipykernel

In [1]:
print('hello123')

hello123


In [ ]:
!pip install --pre torch torchvision torchaudio --index-url https://download.pytorch.org/whl/nightly/cu128

In [2]:
import torch

print(torch.__version__)
print(torch.cuda.is_available())
print(torch.cuda.get_device_name())

2.12.0.dev20260329+cu128
True
NVIDIA GeForce RTX 5060 Ti


In [5]:
!pip -q install -U "transformers>=4.43.2,<5.0.0" "accelerate>=0.34.2" "peft>=0.13.2" "bitsandbytes>=0.43.3" datasets pillow pandas --upgrade 

# 데이터 준비

개발에 필요한 데이터를 준비합니다.

- train.csv, train 폴더
- test.csv, test 폴더
- sample_submission.csv

데이터를 압축 해제하는데 몇 분 정도의 시간이 소요됩니다.

# 라이브러리, 데이터, 설정

In [3]:
import os, re, math, random
import pandas as pd
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from dataclasses import dataclass
import torch
from typing import Dict, List, Any
from transformers import (
    AutoModelForVision2Seq,
    AutoProcessor,
    BitsAndBytesConfig,
    get_linear_schedule_with_warmup
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from tqdm import tqdm

# 이미지 로드 시 픽셀 제한 해제
Image.MAX_IMAGE_PIXELS = None

# 디바이스 GPU 우선 사용 설정
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

# 사전 학습 모델 정의
MODEL_ID = "Qwen/Qwen2.5-VL-3B-Instruct"
# 수정 6
IMAGE_SIZE = 256 # 이전 : 384
MAX_NEW_TOKENS = 8
SEED = 42
random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

# 데이터셋 로드
train_df = pd.read_csv("train.csv")
test_df  = pd.read_csv("test.csv")

# 학습데이터 200개만 추출
# train_df = train_df.sample(n=200, random_state=SEED).reset_index(drop=True)
# 수정 4 (학습데이터 전체로 확대)
train_df.reset_index(drop=True)

/usr/local/lib/python3.10/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device: cuda


,id,path,question,a,b,c,d,answer
0,train_0001.jpg,train/train_0001.jpg,사진 속 흰색 텀블러의 재질은 무엇인가요?,유리,금속,종이,플라스틱,b
1,train_0002.jpg,train/train_0002.jpg,사진에 보이는 재활용 가능한 병의 종류는 무엇인가요?,금속 캔,유리 병,플라스틱 병,종이 팩,b
2,train_0003.jpg,train/train_0003.jpg,사진 속 재활용품 중 플라스틱 재질인 것은 무엇인가요?,껌 상자,헤어끈,음료 컵과 뚜껑,종이컵 홀더,c
3,train_0004.jpg,train/train_0004.jpg,사진에 보이는 '빅파이' 과자 상자의 재활용 분류는 무엇인가요?,종이류,유리류,플라스틱류,금속류,a
4,train_0005.jpg,train/train_0005.jpg,사진 속 음료 용기의 재활용 분류는 무엇인가요?,종이,플라스틱,금속,유리,b
...,...,...,...,...,...,...,...,...
5068,train_5069.jpg,train/train_5069.jpg,사진에 보이는 재활용품 중 금속 재질은 무엇인가요?,흰색 플라스틱 병 두 개,나무 바닥,녹색 캔 한 개,종이봉투 한 개,c
5069,train_5070.jpg,train/train_5070.jpg,사진에 보이는 재활용품은 무엇인가요?,유리병 2개,플라스틱 병 2개,알루미늄 캔 2개,종이컵 2개,c
5070,train_5071.jpg,train/train_5071.jpg,사진에 보이는 재활용품 중 플라스틱 병과 종이 상자의 개수는 각각 몇 개인가요?,"플라스틱 병 2개, 종이 상자 1개","플라스틱 병 1개, 종이 상자 1개","플라스틱 병 1개, 종이 상자 2개","플라스틱 병 2개, 종이 상자 2개",b
5071,train_5072.jpg,train/train_5072.jpg,사진에 보이는 재활용품 중에서 어떤 재질의 물건이 가장 크게 보이나요?,금속,유리,종이,플라스틱,d


# 모델, Processor

7.5GB 정도의 모델 다운로드가 진행됩니다. 10~20분 정도가 소요됩니다.

#### 실습 참고 내용

    챕터 5-1 PEFT(파라미터 효율적 튜닝)
    - LoRA 구현 : LoraConfig()

In [4]:
# 양자화
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)

# 프로세서
processor = AutoProcessor.from_pretrained(
    MODEL_ID,
    min_pixels=IMAGE_SIZE*IMAGE_SIZE,
    max_pixels=IMAGE_SIZE*IMAGE_SIZE,
    trust_remote_code=True,
)

# 사전학습 모델
base_model = AutoModelForVision2Seq.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)

# 양자화 모델로 로드
base_model = prepare_model_for_kbit_training(base_model)
base_model.gradient_checkpointing_enable()

# LoRA 세팅
# 수정 5
lora_config = LoraConfig(
    r=16,               # 기존 8에서 32로 증가시켜 더 복잡한 특징을 학습
    lora_alpha=32,      # r의 2배 값 적용
    lora_dropout=0.05,
    bias="none",
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"], # LLM뿐만 아니라 Vision Encoder의 선형 레이어까지 모두 타겟팅
    task_type="CAUSAL_LM",
    use_dora=True       # [핵심] 기존 LoRA 파라미터 수로 전체 파인튜닝 급의 성능을 내는 DoRA 활성화
)

# PEFT 모델 생성
model = get_peft_model(base_model, lora_config)
model.print_trainable_parameters()

The image processor of type `Qwen2VLImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. Note that this behavior will be extended to all models in a future release.
/usr/local/lib/python3.10/dist-packages/transformers/models/auto/modeling_auto.py:2284: FutureWarning: The class `AutoModelForVision2Seq` is deprecated and will be removed in v5.0. Please use `AutoModelForImageTextToText` instead.
  warnings.warn(
Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]/usr/local/lib/python3.10/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
Loading checkpoint shards: 100%|██████████| 2/

trainable params: 38,444,800 || all params: 3,793,067,776 || trainable%: 1.0136


# 프롬프트 템플릿

#### 실습 참고 내용

    챕터 5-1 PEFT(파라미터 효율적 튜닝)
    - 프롬프트 템플릿 : convert_to_chatml(), formatting_prompts_func()

In [5]:
# 모델 지시사항
SYSTEM_INSTRUCT = (
    "You are a helpful visual question answering assistant. "
    "Answer using exactly one letter among a, b, c, or d. No explanation."
)

# 프롬프트
def build_mc_prompt(question, a, b, c, d):
    return (
        f"{question}\n"
        f"(a) {a}\n(b) {b}\n(c) {c}\n(d) {d}\n\n"
        "정답을 반드시 a, b, c, d 중 하나의 소문자 한 글자로만 출력하세요."
    )

# Custom Dataset, Collator

#### 실습 참고 내용

    챕터 1-2 MLP 구현
    - TensorDataset()

    챕터 5-2 데이터 생성 및 파인튜닝 (향후 학습 분량)
    - IntentDataset()

In [6]:
# 커스텀 데이터셋
class VQAMCDataset(Dataset):
    def __init__(self, df, processor, train=True):
        self.df = df.reset_index(drop=True)
        self.processor = processor
        self.train = train

    def __len__(self): return len(self.df)

    def __getitem__(self, i):
        row = self.df.iloc[i]
        img = Image.open(row["path"]).convert("RGB")

        q = str(row["question"])
        a, b, c, d = str(row["a"]), str(row["b"]), str(row["c"]), str(row["d"])
        user_text = build_mc_prompt(q, a, b, c, d)

        messages = [
            {"role":"system","content":[{"type":"text","text":SYSTEM_INSTRUCT}]},
            {"role":"user","content":[
                {"type":"image","image":img},
                {"type":"text","text":user_text}
            ]}
        ]
        if self.train:
            gold = str(row["answer"]).strip().lower()
            messages.append({"role":"assistant","content":[{"type":"text","text":gold}]})

        return {"messages": messages, "image": img}

# 데이터 콜레이터
@dataclass
class DataCollator:
    processor: Any
    train: bool = True

    def __call__(self, batch):
        texts, images = [], []
        for sample in batch:
            messages = sample["messages"]
            img = sample["image"]

            text = self.processor.apply_chat_template(
                messages,
                tokenize=False,
                add_generation_prompt=False
            )
            texts.append(text)
            images.append(img)

        enc = self.processor(
            text=texts,
            images=images,
            padding=True,
            return_tensors="pt"
        )

        if self.train:
            enc["labels"] = enc["input_ids"].clone()

        return enc


# DataLoader

#### 실습 참고 내용

    챕터 3-1 Transfer Learning 기반의 CNN 모델 학습
    - 데이터로더 정의 : DataLoader()

In [7]:
# 검증용 데이터 분리
split = int(len(train_df)*0.9)
train_subset, valid_subset = train_df.iloc[:split], train_df.iloc[split:]

# VQAMCDataset 형태로 변환
train_ds = VQAMCDataset(train_subset, processor, train=True)
valid_ds = VQAMCDataset(valid_subset, processor, train=True)

## 수정 1
# 데이터로더
train_loader = DataLoader(
    train_ds, 
    batch_size=1,    # VRAM 16GB를 활용해 물리적 배치 사이즈를 늘림
    shuffle=True,
    collate_fn=DataCollator(processor, True), 
    num_workers=2,   # 8코어 CPU 중 4개를 데이터 로딩에 병렬 할당! (SSD의 빠른 속도 활용)
    pin_memory=True, # 2666MHz RAM 환경에서 VRAM으로의 전송 속도를 높임
    drop_last=True
)

valid_loader = DataLoader(
    valid_ds, 
    batch_size=2, 
    shuffle=False, 
    collate_fn=DataCollator(processor, True), 
    num_workers=4, 
    pin_memory=True
)

# fine-tuning

- 200개만 학습 : 10~20분 소요

#### 실습 참고 내용

    챕터 1-2 MLP 구현
    - 모델 정의 : SimpleMLP(), SequentialMLP()

    챕터 3-1 Transfer Learning 기반의 CNN 모델 학습
    - 학습 루프 : 문제 6: 모델 학습을 위한 반복문
    - 추론 : with torch.no_grad(), model.eval()

In [8]:
from tqdm.auto import tqdm

# 수정 2
import bitsandbytes as bnb
model = model.to(device)
GRAD_ACCUM = 4

# 옵티마이저, 학습 스케줄러
# 수정 3
optimizer = bnb.optim.PagedAdamW8bit(model.parameters(), lr=1e-4)

num_training_steps = 1 * math.ceil(len(train_loader)/GRAD_ACCUM)
scheduler = get_linear_schedule_with_warmup(optimizer, int(num_training_steps*0.03), num_training_steps)

# 스케일러
scaler = torch.cuda.amp.GradScaler(enabled=True)

# 학습 루프
global_step = 0
for epoch in range(2):
    running = 0.0
    progress_bar = tqdm(train_loader, desc=f"Epoch {epoch+1} [train]", unit="batch")
    for step, batch in enumerate(progress_bar, start=1):
        batch = {k:v.to(device) for k,v in batch.items()}
        with torch.cuda.amp.autocast(dtype=torch.bfloat16):
            outputs = model(**batch)
            loss = outputs.loss / GRAD_ACCUM

        scaler.scale(loss).backward()
        running += loss.item()

        if step % GRAD_ACCUM == 0:
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)
            scheduler.step()
            global_step += 1

            # 수정 7 (VRAM 찌꺼기 강제 청소)
            torch.cuda.empty_cache()

            avg_loss = running / GRAD_ACCUM
            progress_bar.set_postfix({"loss": f"{avg_loss:.3f}"})
            running = 0.0

    model.eval()
    val_loss = 0.0
    val_steps = 0
    with torch.no_grad(), torch.cuda.amp.autocast(dtype=torch.bfloat16):
        for vb in tqdm(valid_loader, desc=f"Epoch {epoch+1} [valid]", unit="batch"):
            vb = {k:v.to(device) for k,v in vb.items()}
            val_loss += model(**vb).loss.item()
            val_steps += 1
    print(f"[Epoch {epoch+1}] valid loss {val_loss/val_steps:.4f}")
    model.train()

# 모델 저장
SAVE_DIR = "/content/qwen2_5_vl_3b_lora"
model.save_pretrained(SAVE_DIR)
processor.save_pretrained(SAVE_DIR)
print("Saved:", SAVE_DIR)


/tmp/ipykernel_30497/2714515782.py:16: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=True)
Epoch 1 [train]:   0%|          | 0/4565 [00:00<?, ?batch/s]/tmp/ipykernel_30497/2714515782.py:25: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(dtype=torch.bfloat16):
Epoch 1 [train]: 100%|██████████| 4565/4565 [2:25:25<00:00,  1.91s/batch, loss=0.794]  
/tmp/ipykernel_30497/2714515782.py:49: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.no_grad(), torch.cuda.amp.autocast(dtype=torch.bfloat16):
Epoch 1 [valid]: 100%|██████████| 254/254 [04:12<00:00,  1.01batch/s]


[Epoch 1] valid loss 3.5559


Epoch 2 [train]:   0%|          | 0/4565 [00:00<?, ?batch/s]/usr/local/lib/python3.10/dist-packages/torch/utils/checkpoint.py:235: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  check_backward_validity(args)
`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`...
Epoch 2 [valid]: 100%|██████████| 254/254 [04:17<00:00,  1.01s/batch]


[Epoch 2] valid loss 3.5562
Saved: /content/qwen2_5_vl_3b_lora


# inference

30분~1시간 소요

#### 실습 참고 내용

    챕터4-1 RAG 기반 Customer Service AI 에이전트 개발
    - 데이터 파서 : langchain_core.output_parsers(), StrOutputParser()

    챕터 3-1 Transfer Learning 기반의 CNN 모델 학습
    - 학습 루프 : 문제 6: 모델 학습을 위한 반복문
    - 추론 : with torch.no_grad(), model.eval()

In [10]:
# VRAM 정상화
import torch
torch.cuda.empty_cache()

In [12]:
# 1. 테스트용 DataLoader 준비 (병렬 처리 및 배치 묶기)
# 학습 때 만들어둔 클래스를 재활용하여 train=False 모드로 생성합니다.
test_ds = VQAMCDataset(test_df, processor, train=False)

# 추론은 학습(역전파)이 없으므로 VRAM을 적게 차지합니다. 
# batch_size를 8로 늘려 한 번에 8장씩 예측합니다. (OOM 발생 시 4로 조절)
test_loader = DataLoader(
    test_ds, 
    batch_size=8, 
    shuffle=False, 
    collate_fn=DataCollator(processor, train=False), 
    num_workers=4,   # CPU 워커를 4개 할당하여 이미지 전처리 병목 해결
    pin_memory=True
)

# 2. 모델 평가 모드 및 리스트 초기화
model.eval()
preds = []

# (빈칸을 왼쪽에 채우도록 설정)
processor.tokenizer.padding_side = "left"

# 3. 최적화된 추론 루프
# 혼합 정밀도(bfloat16)를 적용하여 연산 속도와 VRAM 효율을 극대화합니다.
with torch.no_grad(), torch.amp.autocast('cuda', dtype=torch.bfloat16):
    for batch in tqdm(test_loader, desc="Inference", unit="batch"):
        # 배치 데이터를 GPU로 이동
        inputs = {k: v.to(device) for k, v in batch.items()}
        
        # 모델 생성 (한 번에 8개씩 생성)
        out_ids = model.generate(
            **inputs, 
            max_new_tokens=2, 
            do_sample=False,
            eos_token_id=processor.tokenizer.eos_token_id
        )
        
        # 입력된 프롬프트 길이를 파악하여, 모델이 새로 생성한 답변(a,b,c,d) 토큰만 잘라냄
        input_len = inputs["input_ids"].shape[1]
        generated_ids = out_ids[:, input_len:]
        
        # 디코딩
        output_texts = processor.batch_decode(generated_ids, skip_special_tokens=True)
        
        # 파서 적용하여 결과 저장
        for text in output_texts:
            preds.append(extract_choice(text))

# 4. 제출 파일 생성
submission = pd.DataFrame({"id": test_df["id"], "answer": preds})
submission.to_csv("content/submission.csv", index=False)
print("Saved content/submission.csv")

Inference:   0%|          | 0/635 [00:00<?, ?batch/s]

Inference: 100%|██████████| 635/635 [24:47<00:00,  2.34s/batch]

Saved content/submission.csv


In [13]:
# 모델 응답 예시
print(output_text)

system
You are a helpful visual question answering assistant. Answer using exactly one letter among a, b, c, or d. No explanation.
user
사진 속 재활용품 상자 안에 주로 어떤 재질의 재활용품이 들어 있나요?
(a) 금속
(b) 유리
(c) 플라스틱
(d) 종이

정답을 반드시 a, b, c, d 중 하나의 소문자 한 글자로만 출력하세요.
assistant
c
